In [ ]:
# Install dependencies if needed (uncomment in Databricks)
# %pip install -r ../requirements.txt
# dbutils.library.restartPython()

In [ ]:
import sys
import time
from pprint import pprint

# Add parent directory to path for imports
sys.path.insert(0, '..')

# Import agent and MLflow setup
from multiAgentSystem.agents.supervisor import supervisor_node
from experiments.mlflow_setup import (
    setup_experiment,
    create_agent_run,
    log_agent_metrics,
    log_state_snapshot,
    enable_autologging
)
from experiments.mock_data import SUPERVISOR_TEST_STATES, get_test_state

print("✓ Imports successful")

In [ ]:
# Setup MLflow experiment
enable_autologging()
experiment_id = setup_experiment("supervisor")
print(f"Experiment ID: {experiment_id}")

In [ ]:
def run_supervisor_test(scenario_name: str, verbose: bool = True):
    """
    Run a single supervisor test scenario with MLflow tracking.
    
    Args:
        scenario_name: Name of the scenario from SUPERVISOR_TEST_STATES
        verbose: Whether to print results
        
    Returns:
        Tuple of (result, passed, latency_ms)
    """
    scenario = get_test_state("supervisor", scenario_name)
    test_state = scenario["state"].copy()
    expected = scenario["expected"]
    
    with create_agent_run("supervisor", scenario=scenario_name) as run:
        # Log input state
        log_state_snapshot(test_state, prefix="input")
        
        # Run the agent
        start_time = time.time()
        try:
            result = supervisor_node(test_state)
            success = True
            error = None
        except Exception as e:
            result = {"error": str(e)}
            success = False
            error = str(e)
        latency_ms = (time.time() - start_time) * 1000
        
        # Log metrics
        log_agent_metrics(
            latency_ms=latency_ms,
            success=success,
            additional_metrics={
                "input_iteration": test_state.get("iteration", 0),
                "input_confidence": test_state.get("confidence", 0.0),
            }
        )
        
        # Log output
        log_state_snapshot(result, prefix="output")
        
        # Check expected outcome
        passed = True
        if "next_action" in expected:
            actual_action = result.get("next_action", "")
            if actual_action != expected["next_action"]:
                passed = False
                if verbose:
                    print(f"  ⚠️ Expected next_action='{expected['next_action']}', got '{actual_action}'")
        
        import mlflow
        mlflow.log_metric("test_passed", 1.0 if passed else 0.0)
        
        if verbose:
            status = "✅ PASSED" if passed else "❌ FAILED"
            print(f"\n{status} - {scenario_name}")
            print(f"  Description: {scenario['description']}")
            print(f"  Latency: {latency_ms:.2f}ms")
            print(f"  Result: next_action='{result.get('next_action')}', iteration={result.get('iteration')}")
            if result.get("supervisor_rationale"):
                print(f"  Rationale: {result.get('supervisor_rationale')[:100]}...")
        
        return result, passed, latency_ms

## Test 1: Initial State

First iteration with no prior work. Should route to `reasoning`.

In [ ]:
result_1, passed_1, latency_1 = run_supervisor_test("initial_state")

## Test 2: After Summarization

Reasoning completed with draft. Should route to `critic`.

In [ ]:
result_2, passed_2, latency_2 = run_supervisor_test("after_summarization")

## Test 3: Critic Approved with High Confidence

Critic approved and confidence >= threshold. Should route to `end`.

In [ ]:
result_3, passed_3, latency_3 = run_supervisor_test("critic_approved_high_confidence")

## Test 4: Critic Rejected / Low Confidence

Critic rejected or confidence too low. Should route to `reasoning` for more work.

In [ ]:
result_4, passed_4, latency_4 = run_supervisor_test("critic_rejected_low_confidence")

## Test 5: Maximum Iterations Reached

Hit iteration limit. Should force `end` regardless of other factors.

In [ ]:
result_5, passed_5, latency_5 = run_supervisor_test("max_iterations_reached")

## Summary

In [ ]:
# Summary of all tests
print("=" * 60)
print("SUPERVISOR AGENT TEST SUMMARY")
print("=" * 60)

tests = [
    ("initial_state", passed_1, latency_1),
    ("after_summarization", passed_2, latency_2),
    ("critic_approved_high_confidence", passed_3, latency_3),
    ("critic_rejected_low_confidence", passed_4, latency_4),
    ("max_iterations_reached", passed_5, latency_5),
]

total_passed = sum(1 for _, passed, _ in tests if passed)
avg_latency = sum(lat for _, _, lat in tests) / len(tests)

for name, passed, latency in tests:
    status = "✅" if passed else "❌"
    print(f"  {status} {name}: {latency:.2f}ms")

print("=" * 60)
print(f"Total: {total_passed}/{len(tests)} passed")
print(f"Average latency: {avg_latency:.2f}ms")
print("=" * 60)